# Notebook 05 — Add Goalscorer Features (Path A Phase 2)

**Goal:** Load `goalscorers.csv`, explore it, and create **new features** from past scoring patterns.

**Input files:**
- `data/raw/goalscorers.csv`
- `data/processed/matches_2021_onwards.csv`

**Output file:** `data/processed/matches_ready_v2.csv`

**Important:** We only use scoring history from **past matches** (no data leakage).

# Step 1 — Load goalscorers.csv

This file lists **every goal** in international matches:
who scored, when, penalties, own goals.

Same Kaggle download as `results.csv` — file should be in `data/raw/`.

In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
SCORERS_PATH = PROJECT_ROOT / "data" / "raw" / "goalscorers.csv"
MATCHES_PATH = PROJECT_ROOT / "data" / "processed" / "matches_2021_onwards.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "matches_ready_v2.csv"

# Load goalscorer events
scorers = pd.read_csv(SCORERS_PATH)
scorers["date"] = pd.to_datetime(scorers["date"])

print("Goalscorer rows:", len(scorers))
print("Columns:", list(scorers.columns))
scorers.head()

Goalscorer rows: 47727
Columns: ['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False


# Step 2 — Understand goalscorers Columns

| Column | Meaning |
|--------|--------|
| `date`, `home_team`, `away_team` | Which match |
| `team` | Team that scored (or own-goal team label) |
| `scorer` | Player name |
| `minute` | Minute of goal |
| `own_goal` | TRUE if own goal |
| `penalty` | TRUE if penalty |

In [3]:
# How many goal events since 2021?
recent_scorers = scorers[scorers["date"] >= "2021-01-01"]
print("Goal events 2021+:", len(recent_scorers))

# Sample: one match can have many rows (one row per goal)
sample_match = scorers[
    (scorers["date"] == "2022-11-22")
    & (scorers["home_team"] == "Argentina")
    & (scorers["away_team"] == "Saudi Arabia")
]
print("\nExample match goals:")
print(sample_match[["team", "scorer", "minute", "penalty"]])

Goal events 2021+: 7659

Example match goals:
               team            scorer  minute  penalty
42906     Argentina      Lionel Messi    10.0     True
42907  Saudi Arabia   Saleh Al-Shehri    48.0    False
42908  Saudi Arabia  Salem Al-Dawsari    53.0    False


# Step 3 — Rolling Goal Form (from match results)

Average goals **scored** and **conceded** in each team's last 5 matches.

`.shift(1)` = use only previous matches (not today's match — avoids leakage).

In [4]:
matches = pd.read_csv(MATCHES_PATH)
matches["date"] = pd.to_datetime(matches["date"])

# Build one row per team per match (long format)
home_log = matches[["date", "home_team", "away_team", "home_score", "away_score"]].rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "goals_for",
        "away_score": "goals_against",
    }
)
away_log = matches[["date", "home_team", "away_team", "home_score", "away_score"]].rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "goals_for",
        "home_score": "goals_against",
    }
)
team_log = pd.concat([home_log, away_log], ignore_index=True).sort_values(["team", "date"])

# Rolling average from PREVIOUS 5 games only
for col in ["goals_for", "goals_against"]:
    team_log[f"roll_{col}_5"] = team_log.groupby("team")[col].transform(
        lambda s: s.shift(1).rolling(5, min_periods=1).mean()
    )

print("Team form sample:")
team_log[team_log["team"] == "Argentina"].tail(3)

Team form sample:


,date,team,opponent,goals_for,goals_against,roll_goals_for_5,roll_goals_against_5
5619,2026-06-06,Argentina,Honduras,2.0,0.0,3.2,0.2
5658,2026-06-09,Argentina,Iceland,3.0,0.0,3.4,0.2
5701,2026-06-16,Argentina,Algeria,3.0,0.0,2.8,0.2


# Step 4 — Goalscorer Features (unique scorers + penalties)

From `goalscorers.csv` we compute per team per match:
- **unique_scorers** — how many different players scored
- **penalty_goals** — how many penalty goals

Then rolling averages from past 5 matches.

In [5]:
# Flag penalty goals
scorers["is_penalty"] = scorers["penalty"].astype(str).str.upper().eq("TRUE")

# Count unique scorers and penalties per team per match
uniq = scorers.groupby(["date", "home_team", "away_team", "team"])["scorer"].nunique().reset_index(name="unique_scorers")
pen = scorers.groupby(["date", "home_team", "away_team", "team"])["is_penalty"].sum().reset_index(name="penalty_goals")
team_stats = uniq.merge(pen, on=["date", "home_team", "away_team", "team"], how="left")
team_stats["penalty_goals"] = team_stats["penalty_goals"].fillna(0)

# Team timeline for rolling scorer stats
timeline = team_stats[["date", "team", "unique_scorers", "penalty_goals"]].sort_values(["team", "date"])
timeline["roll_unique_scorers_5"] = timeline.groupby("team")["unique_scorers"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)
timeline["roll_penalty_goals_5"] = timeline.groupby("team")["penalty_goals"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

print("Goalscorer timeline sample:")
timeline[timeline["team"] == "Argentina"].tail(3)

Goalscorer timeline sample:


,date,team,unique_scorers,penalty_goals,roll_unique_scorers_5,roll_penalty_goals_5
22079,2025-06-10,Argentina,1,0,1.6,0.0
22172,2025-09-04,Argentina,2,0,1.6,0.0
22684,2026-06-16,Argentina,1,0,1.8,0.0


# Step 5 — Merge All Features Into One Table

Attach home team form + away team form to each match row.

In [6]:
# Merge rolling goal form
home_form = team_log.rename(
    columns={
        "team": "home_team",
        "roll_goals_for_5": "home_roll_goals_for_5",
        "roll_goals_against_5": "home_roll_goals_against_5",
    }
)[["date", "home_team", "home_roll_goals_for_5", "home_roll_goals_against_5"]]

away_form = team_log.rename(
    columns={
        "team": "away_team",
        "roll_goals_for_5": "away_roll_goals_for_5",
        "roll_goals_against_5": "away_roll_goals_against_5",
    }
)[["date", "away_team", "away_roll_goals_for_5", "away_roll_goals_against_5"]]

df = matches.merge(home_form, on=["date", "home_team"], how="left")
df = df.merge(away_form, on=["date", "away_team"], how="left")

# Merge goalscorer rolling stats
home_gs = timeline.rename(
    columns={
        "team": "home_team",
        "roll_unique_scorers_5": "home_roll_unique_scorers_5",
        "roll_penalty_goals_5": "home_roll_penalty_goals_5",
    }
)[["date", "home_team", "home_roll_unique_scorers_5", "home_roll_penalty_goals_5"]]

away_gs = timeline.rename(
    columns={
        "team": "away_team",
        "roll_unique_scorers_5": "away_roll_unique_scorers_5",
        "roll_penalty_goals_5": "away_roll_penalty_goals_5",
    }
)[["date", "away_team", "away_roll_unique_scorers_5", "away_roll_penalty_goals_5"]]

df = df.merge(home_gs, on=["date", "home_team"], how="left")
df = df.merge(away_gs, on=["date", "away_team"], how="left")

print("Merged shape:", df.shape)
df.head()

Merged shape: (5723, 17)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_roll_goals_for_5,home_roll_goals_against_5,away_roll_goals_for_5,away_roll_goals_against_5,home_roll_unique_scorers_5,home_roll_penalty_goals_5,away_roll_unique_scorers_5,away_roll_penalty_goals_5
0,2021-01-12,United Arab Emirates,Iraq,0.0,0.0,Friendly,Dubai,United Arab Emirates,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-18,Kuwait,Palestine,0.0,1.0,Friendly,Kuwait City,Kuwait,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-19,Dominican Republic,Puerto Rico,0.0,1.0,Friendly,Santo Domingo,Dominican Republic,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-01-22,Guatemala,Puerto Rico,1.0,0.0,Friendly,Guatemala City,Guatemala,False,NaN,NaN,1.0,0.0,NaN,NaN,NaN,NaN
4,2021-01-25,Dominican Republic,Serbia,0.0,0.0,Friendly,Santo Domingo,Dominican Republic,False,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN


# Step 6 — Create Target and Final Feature List

Same target as before: **Win / Draw / Loss** for home team.

Fill missing rolling values with 0 (team's first matches have no history).

In [7]:
def get_home_result(row):
    """Returns Win, Draw, or Loss for the home team."""
    if row["home_score"] > row["away_score"]:
        return "Win"
    elif row["home_score"] == row["away_score"]:
        return "Draw"
    else:
        return "Loss"


df["home_result"] = df.apply(get_home_result, axis=1)
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["is_neutral"] = df["neutral"].astype(int)

NEW_NUMERIC_FEATURES = [
    "home_roll_goals_for_5",
    "home_roll_goals_against_5",
    "away_roll_goals_for_5",
    "away_roll_goals_against_5",
    "home_roll_unique_scorers_5",
    "away_roll_unique_scorers_5",
    "home_roll_penalty_goals_5",
    "away_roll_penalty_goals_5",
]

FEATURE_COLUMNS = [
    "home_team",
    "away_team",
    "tournament",
    "is_neutral",
    "year",
    "month",
] + NEW_NUMERIC_FEATURES

TARGET_COLUMN = "home_result"

# Fill missing rolling stats
df[NEW_NUMERIC_FEATURES] = df[NEW_NUMERIC_FEATURES].fillna(0)

model_df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()

print("Modeling dataset shape:", model_df.shape)
print("\nNew features added:")
for col in NEW_NUMERIC_FEATURES:
    print(" -", col)

model_df.head()

Modeling dataset shape: (5723, 15)

New features added:
 - home_roll_goals_for_5
 - home_roll_goals_against_5
 - away_roll_goals_for_5
 - away_roll_goals_against_5
 - home_roll_unique_scorers_5
 - away_roll_unique_scorers_5
 - home_roll_penalty_goals_5
 - away_roll_penalty_goals_5


,home_team,away_team,tournament,is_neutral,year,month,home_roll_goals_for_5,home_roll_goals_against_5,away_roll_goals_for_5,away_roll_goals_against_5,home_roll_unique_scorers_5,away_roll_unique_scorers_5,home_roll_penalty_goals_5,away_roll_penalty_goals_5,home_result
0,United Arab Emirates,Iraq,Friendly,0,2021,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Draw
1,Kuwait,Palestine,Friendly,0,2021,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Loss
2,Dominican Republic,Puerto Rico,Friendly,0,2021,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Loss
3,Guatemala,Puerto Rico,Friendly,0,2021,1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Win
4,Dominican Republic,Serbia,Friendly,0,2021,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,Draw


# Step 7 — Save Version 2 Dataset

In [8]:
model_df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH.resolve())
print("Rows saved:", len(model_df))
print("\nNext: run notebook 06 to retrain and compare accuracy vs v1 (~52%).")

Saved to: C:\Users\HP-15\Desktop\worldcup_predictor\data\processed\matches_ready_v2.csv
Rows saved: 5723

Next: run notebook 06 to retrain and compare accuracy vs v1 (~52%).
